In [2]:
"""
Quick VALUE & QUANTITY CONSISTENCY CHECKS —  Module 4
Checks:
- Negative values
- Zero values (warning)
- Invalid dates
- Future dates
"""

import glob
import pandas as pd
from pathlib import Path

# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────
BASE_DIR    = Path.cwd().parent
EXPORTS_DIR = BASE_DIR / "Dataset" / "Exports"

# ──────────────────────────────────────────────────────────────
# LOAD (same logic as previous modules)
# ──────────────────────────────────────────────────────────────
def load_raw(filepath):
    df = pd.read_csv(filepath, skiprows=1, dtype=str)
    df.columns = df.columns.str.strip()

    if "Period" in df.columns:
        df = df[pd.to_datetime(df["Period"], errors="coerce").notna() |
                (df["Period"].astype(str).str.strip() == "")]

    df["_source_file"] = Path(filepath).name
    return df


def load_all(files):
    frames = []
    errors = []
    for f in files:
        try:
            frames.append(load_raw(f))
        except Exception as e:
            errors.append({"file": Path(f).name, "error": str(e)})

    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return combined, errors


# ──────────────────────────────────────────────────────────────
# VALIDATION LOGIC
# ──────────────────────────────────────────────────────────────
def validate_ranges(raw):

    df = raw.copy()
    n = len(df)

    results = {
        "summary": [],
        "examples": {}
    }

    # Convert types safely
    df["_value"]  = pd.to_numeric(df.get("Value ($)"), errors="coerce")
    df["_qty"]    = pd.to_numeric(df.get("Quantity"), errors="coerce")
    df["_period"] = pd.to_datetime(df.get("Period"), errors="coerce")

    # ═══════════════════════════════════════════════════════════
    # 1. NEGATIVE VALUES
    # ═══════════════════════════════════════════════════════════
    for col_internal, col_name in [("_value", "Value ($)"),
                                   ("_qty", "Quantity")]:

        mask = df[col_internal] < 0
        count = mask.sum()
        pct = round(count / n * 100, 2) if n else 0

        severity = "🔴 CRITICAL" if count > 0 else "✅ OK"

        results["summary"].append({
            "check": f"Negative values in {col_name}",
            "count": int(count),
            "pct_of_total": pct,
            "severity": severity
        })

        if count > 0:
            results["examples"][f"negative_{col_name}"] = raw[mask].head(5)

    # ═══════════════════════════════════════════════════════════
    # 2. ZERO VALUES (WARNING ONLY)
    # ═══════════════════════════════════════════════════════════
    for col_internal, col_name in [("_value", "Value ($)"),
                                   ("_qty", "Quantity")]:

        mask = df[col_internal] == 0
        count = mask.sum()
        pct = round(count / n * 100, 2) if n else 0

        severity = "🟡 WARN" if count > 0 else "✅ OK"

        results["summary"].append({
            "check": f"Zero values in {col_name}",
            "count": int(count),
            "pct_of_total": pct,
            "severity": severity
        })

        if count > 0:
            results["examples"][f"zero_{col_name}"] = raw[mask].head(5)

    # ═══════════════════════════════════════════════════════════
    # 3. INVALID DATES
    # ═══════════════════════════════════════════════════════════
    invalid_mask = df["_period"].isna() & df["Period"].notna()
    count = invalid_mask.sum()
    pct = round(count / n * 100, 2) if n else 0

    severity = "🔴 CRITICAL" if count > 0 else "✅ OK"

    results["summary"].append({
        "check": "Invalid / Unparseable dates",
        "count": int(count),
        "pct_of_total": pct,
        "severity": severity
    })

    if count > 0:
        results["examples"]["invalid_dates"] = raw[invalid_mask].head(5)

    # ═══════════════════════════════════════════════════════════
    # 4. FUTURE DATES
    # ═══════════════════════════════════════════════════════════
    today = pd.Timestamp.today()
    future_mask = df["_period"] > today
    count = future_mask.sum()
    pct = round(count / n * 100, 2) if n else 0

    severity = "🔴 CRITICAL" if count > 0 else "✅ OK"

    results["summary"].append({
        "check": "Future dates",
        "count": int(count),
        "pct_of_total": pct,
        "severity": severity
    })

    if count > 0:
        results["examples"]["future_dates"] = raw[future_mask].head(5)

    return results

def zero_percentage_by_group(raw):

    df = raw.copy()

    # Convert numeric safely
    df["_value"] = pd.to_numeric(df.get("Value ($)"), errors="coerce")
    df["_qty"]   = pd.to_numeric(df.get("Quantity"), errors="coerce")

    # Flags
    df["_zero_value"] = df["_value"] == 0
    df["_zero_qty"]   = df["_qty"] == 0
    df["_zero_both"]  = df["_zero_value"] & df["_zero_qty"]

    # At least one zero (value OR qty)
    df["_any_zero"] = df["_zero_value"] | df["_zero_qty"]

    summary = (
        df.groupby("Commodity")
          .agg(
              total_rows          = ("_value", "size"),
              total_rows_w_zero   = ("_any_zero", "sum"),
              zero_value_cnt      = ("_zero_value", "sum"),
              zero_qty_cnt        = ("_zero_qty", "sum"),
              zero_both_cnt       = ("_zero_both", "sum"),
          )
          .reset_index()
    )

    # Percentages
    summary["pct_zero_value"] = (
        summary["zero_value_cnt"] / summary["total_rows"] * 100
    ).round(2)

    summary["pct_zero_qty"] = (
        summary["zero_qty_cnt"] / summary["total_rows"] * 100
    ).round(2)

    summary["pct_zero_both"] = (
        summary["zero_both_cnt"] / summary["total_rows"] * 100
    ).round(2)

    # Keep clean output
    final_table = summary[
        [
            "Commodity",
            "total_rows",
            "total_rows_w_zero",
            "pct_zero_value",
            "pct_zero_qty",
            "pct_zero_both",
        ]
    ]

    # Sort by severity
    final_table = final_table.sort_values(
        by=["pct_zero_both", "pct_zero_value", "pct_zero_qty"],
        ascending=False
    )

    # Top 10 worst offenders
    final_table = final_table.head(20)

    return final_table
# ──────────────────────────────────────────────────────────────
# PRINT RESULTS
# ──────────────────────────────────────────────────────────────
def print_results(results):

    print("\n" + "="*70)
    print("  MODULE 4 — VALUE & QUANTITY CONSISTENCY CHECKS")
    print("="*70 + "\n")

    for i, r in enumerate(results["summary"], 1):
        print(f"{i}. {r['check']}")
        print(f"   Count:    {r['count']:,} ({r['pct_of_total']}%)")
        print(f"   Severity: {r['severity']}\n")

    print("="*70)
    print("  EXAMPLE ROWS")
    print("="*70 + "\n")

    for key, df_example in results["examples"].items():
        print(f"Example: {key}")
        print("-"*50)

        for idx, row in df_example.iterrows():
            print(f"Row {idx}:")
            for col in df_example.columns:
                val = row[col]
                val_str = "[EMPTY]" if pd.isna(val) else str(val)[:50]
                print(f"   {col:20s}: {val_str}")
            print()
        print()


# ──────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────
if __name__ == "__main__":

    print("\nRunning value & quantity consistency checks...\n")

    export_files = glob.glob(str(EXPORTS_DIR / "**/*.csv"), recursive=True)

    if not export_files:
        print("❌ No files found.")
        exit(1)

    raw, _ = load_all(export_files)

    results = validate_ranges(raw)
    print_results(results)
    zero_table = zero_percentage_by_group(raw)

    print("\n" + "="*70)
    print("ZERO VALUE / ZERO QUANTITY PERCENTAGE TABLE")
    print("="*70)
    print(zero_table.to_string(index=False))


Running value & quantity consistency checks...


  MODULE 4 — VALUE & QUANTITY CONSISTENCY CHECKS

1. Negative values in Value ($)
   Count:    0 (0.0%)
   Severity: ✅ OK

2. Negative values in Quantity
   Count:    0 (0.0%)
   Severity: ✅ OK

3. Zero values in Value ($)
   Count:    1,074 (0.04%)
   Severity: 🟡 WARN

4. Zero values in Quantity
   Count:    1,259,988 (49.67%)
   Severity: 🟡 WARN

5. Invalid / Unparseable dates
   Count:    0 (0.0%)
   Severity: ✅ OK

6. Future dates
   Count:    0 (0.0%)
   Severity: ✅ OK

  EXAMPLE ROWS

Example: zero_Value ($)
--------------------------------------------------
Row 2188:
   Period              : 2025-07-01
   Commodity           : 4911.10.00 - Trade advertising material, commercia
   Province            : Ontario
   Country             : Aruba
   State               : [EMPTY]
   Value ($)           : 0
   Quantity            : 0
   Unit of measure     : Blank
   _source_file        : 07-Exports_ON_2025_Jul.csv

Row 2438:
   Period   